In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime, timedelta, date, timezone
from functools import partial
import numpy as np
import polars as pl

from okx.store import OrderbookStore
from okx.recipes.pillars import prepare_pillars
from okx.recipes.helpers import finalize_binning, early_roll

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite"
)

In [8]:
store.clear_cache()

Cleared all caches


In [9]:
start_date = date(2025, 9, 1)
end_date = date(2025, 10, 1)
dates = [start_date + timedelta(days=i) for i in range((end_date - start_date).days)]
print(dates)

[datetime.date(2025, 9, 1), datetime.date(2025, 9, 2), datetime.date(2025, 9, 3), datetime.date(2025, 9, 4), datetime.date(2025, 9, 5), datetime.date(2025, 9, 6), datetime.date(2025, 9, 7), datetime.date(2025, 9, 8), datetime.date(2025, 9, 9), datetime.date(2025, 9, 10), datetime.date(2025, 9, 11), datetime.date(2025, 9, 12), datetime.date(2025, 9, 13), datetime.date(2025, 9, 14), datetime.date(2025, 9, 15), datetime.date(2025, 9, 16), datetime.date(2025, 9, 17), datetime.date(2025, 9, 18), datetime.date(2025, 9, 19), datetime.date(2025, 9, 20), datetime.date(2025, 9, 21), datetime.date(2025, 9, 22), datetime.date(2025, 9, 23), datetime.date(2025, 9, 24), datetime.date(2025, 9, 25), datetime.date(2025, 9, 26), datetime.date(2025, 9, 27), datetime.date(2025, 9, 28), datetime.date(2025, 9, 29), datetime.date(2025, 9, 30)]


In [ ]:
start = datetime.now()
pillars_lf = prepare_pillars(
    store,
    inst_family="BTC-USD",
    dates=dates,
    binning='10s',
    verbose=True
)
end = datetime.now()
print(f"Time taken: {end - start}")
pillars_df = pillars_lf.collect()
print(f"Time taken to collect: {datetime.now() - end}")

[datetime.date(2025, 9, 1), datetime.date(2025, 9, 2), datetime.date(2025, 9, 3), datetime.date(2025, 9, 4), datetime.date(2025, 9, 5), datetime.date(2025, 9, 6), datetime.date(2025, 9, 7), datetime.date(2025, 9, 8), datetime.date(2025, 9, 9), datetime.date(2025, 9, 10), datetime.date(2025, 9, 11), datetime.date(2025, 9, 12), datetime.date(2025, 9, 13), datetime.date(2025, 9, 14), datetime.date(2025, 9, 15), datetime.date(2025, 9, 16), datetime.date(2025, 9, 17), datetime.date(2025, 9, 18), datetime.date(2025, 9, 19), datetime.date(2025, 9, 20), datetime.date(2025, 9, 21), datetime.date(2025, 9, 22), datetime.date(2025, 9, 23), datetime.date(2025, 9, 24), datetime.date(2025, 9, 25), datetime.date(2025, 9, 26), datetime.date(2025, 9, 27), datetime.date(2025, 9, 28), datetime.date(2025, 9, 29), datetime.date(2025, 9, 30)]
[store] Applying transforms for BTC-USD/SWAP (depth=1, binning=10s, features=['trim', 'strip', 'bin_ff', <function finalize_binning at 0x137a962a0>, 'spread', 'rel_spre

## Replicate futures forward pipeline to track down null column appearance

In [10]:
def get_first_and_last_ts(lf, symbols):
    """Returns a dict: symbol -> dict with first_ms, last_ms, first_dt, last_dt."""
    results = {}
    for symbol in symbols:
        res = (
            lf.filter(pl.col('symbol') == symbol)
            .select([
                pl.col('timeMs').min().alias('first_ms'),
                pl.col('timeMs').max().alias('last_ms'),
            ])
            .collect()
        )
        # If symbol is not present, skip
        if res.height == 0 or res['first_ms'][0] is None:
            results[symbol] = {
                'first_ms': None,
                'last_ms': None,
                'first_dt': None,
                'last_dt': None,
            }
        else:
            first_ms = res['first_ms'][0]
            last_ms = res['last_ms'][0]
            first_dt = datetime.fromtimestamp(first_ms / 1000, tz=timezone.utc)
            last_dt  = datetime.fromtimestamp(last_ms / 1000, tz=timezone.utc)
            results[symbol] = {
                'first_ms': first_ms,
                'last_ms': last_ms,
                'first_dt': first_dt,
                'last_dt': last_dt,
            }
    return results

def null_search(lf, symbols):
    """Returns a dict: symbol -> dict with null_count and the null DataFrame (head 10)."""
    # Determine whether log-space columns are present
    schema_cols = lf.collect_schema().names()
    if 'ln_bid_1_px' in schema_cols and 'ln_ask_1_px' in schema_cols:
        bid_col = 'ln_bid_1_px'
        ask_col = 'ln_ask_1_px'
    else:
        bid_col = 'bid_1_px'
        ask_col = 'ask_1_px'
    # Take the union of baseline and available columns in schema
    collect_cols = ['timeMs', 'symbol', bid_col, ask_col, 'time_bin']
    cols = [col for col in collect_cols if col in schema_cols]
        
    results = {}
    for symbol in symbols:
        null_df = (
            lf.filter(pl.col('symbol') == symbol)
            .filter(pl.col(bid_col).is_null() & pl.col(ask_col).is_null())
            .select(cols).collect()
        )
        null_count = null_df.height
        # Only keep first 10 to avoid giant output
        null_head = null_df.head(10) if null_count > 0 else null_df
        results[symbol] = {
            'null_count': null_count,
            'null_rows': null_head,
        }
    return results

def test_stage(stage: str, features: list, shared_params: dict):
    print(f"\n=== Step {stage}: ===")
    lf = store.get(features=features, cache_name=f'test_cache_{stage}', **shared_params)
    symbols = lf.select('symbol').unique().collect()['symbol'].to_list()
    print("Data fetched")

    minmax = get_first_and_last_ts(lf, symbols)
    print("Timestamps tested")
    nulls = null_search(lf, symbols)
    print("Nulls tested")
    for symbol, data in nulls.items():
        if data['null_count'] > 0:
            print(f"{symbol}: {data['null_count']} null rows")
            print(data['null_rows'])
    return lf, symbols, minmax, nulls

In [11]:
def add_to_side_by_side(side_by_side, stage, symbols, minmax, nulls):
    for symbol in symbols:
        if symbol not in side_by_side:
            side_by_side[symbol] = {}
        side_by_side[symbol][stage] = {
            'first_dt': minmax[symbol]['first_dt'],
            'last_dt': minmax[symbol]['last_dt'],
            'null_count': nulls[symbol]['null_count'],
        }

shared_params = {
    'inst_family': 'BTC-USD',
    'inst_type': 'FUTURES',
    'dates': dates,
    'depth': 1,
    'verbose': True,
    'batch_days': None
}
side_by_side = {}

print("\n=== Step 1: Fetch raw futures data ===")
stage = 'trim_strip'
_, symbols, minmax, nulls = test_stage(stage, ['trim', 'strip'], shared_params)
add_to_side_by_side(side_by_side, stage, symbols, minmax, nulls)

print("\n=== Step 2: After bin_ff (10s with forward fill) ===")
stage = 'bin_ff'
shared_params['binning'] = '10s'
_, symbols, minmax, nulls = test_stage(stage, ['trim', 'strip', 'bin_ff'], shared_params)
add_to_side_by_side(side_by_side, stage, symbols, minmax, nulls)

print("\n=== Step 3: After finalize_binning ===")
stage = 'finalize_binning'
_, symbols, minmax, nulls = test_stage(stage, ['trim', 'strip', 'bin_ff', finalize_binning], shared_params)
add_to_side_by_side(side_by_side, stage, symbols, minmax, nulls)

print("\n=== Step 4: After spread ===")
stage = 'spread'
_, symbols, minmax, nulls = test_stage(stage, ['trim', 'strip', 'bin_ff', finalize_binning, 'spread'], shared_params)
add_to_side_by_side(side_by_side, stage, symbols, minmax, nulls)

print("\n=== Step 5: After rel_spread ===")
stage = 'rel_spread'
_, symbols, minmax, nulls = test_stage(stage, ['trim', 'strip', 'bin_ff', finalize_binning, 'spread', 'rel_spread'], shared_params)
add_to_side_by_side(side_by_side, stage, symbols, minmax, nulls)

print("\n=== Step 6: After tenor ===")
stage = 'tenor'
_, symbols, minmax, nulls = test_stage(stage, ['trim', 'strip', 'bin_ff', finalize_binning, 'spread', 'rel_spread', 'tenor'], shared_params)
add_to_side_by_side(side_by_side, stage, symbols, minmax, nulls)

print("\n=== Step 7: After early_roll (min_time_to_expiry_hours=2) ===")
stage = 'early_roll'
_, symbols, minmax, nulls = test_stage(
    stage,
    ['trim', 'strip', 'bin_ff', finalize_binning, 'spread', 'rel_spread', 'tenor', early_roll(2)],
    shared_params,
)
add_to_side_by_side(side_by_side, stage, symbols, minmax, nulls)

print("\n=== Step 8: After log ===")
stage = 'log'
_, symbols, minmax, nulls = test_stage(
    stage,
    ['trim', 'strip', 'bin_ff', finalize_binning, 'spread', 'rel_spread', 'tenor', early_roll(2), 'log'],
    shared_params,
)
add_to_side_by_side(side_by_side, stage, symbols, minmax, nulls)




=== Step 1: Fetch raw futures data ===

=== Step trim_strip: ===
[store] Applying transforms for BTC-USD/FUTURES (depth=1, binning=None, features=['trim', 'strip'])
  - applied 'trim'
  - applied 'strip'
Data fetched
Timestamps tested
Nulls tested

=== Step 2: After bin_ff (10s with forward fill) ===

=== Step bin_ff: ===
[store] Applying transforms for BTC-USD/FUTURES (depth=1, binning=10s, features=['trim', 'strip', 'bin_ff'])
  - applied 'trim'
  - applied 'strip'
  - applied 'bin_ff'
Data fetched
Timestamps tested
Nulls tested

=== Step 3: After finalize_binning ===

=== Step finalize_binning: ===
[store] Applying transforms for BTC-USD/FUTURES (depth=1, binning=10s, features=['trim', 'strip', 'bin_ff', <function finalize_binning at 0x15f05a020>])
  - applied 'trim'
  - applied 'strip'
  - applied 'bin_ff'
  - applied 'finalize_binning'
Data fetched
Timestamps tested
Nulls tested

=== Step 4: After spread ===

=== Step spread: ===
[store] Applying transforms for BTC-USD/FUTURES (d

In [12]:
print("\n=== Side-by-side summary ===")
# Dynamically print all available stages for each symbol in side_by_side

def format_dt(dt):
    # dt: int milliseconds or a datetime or None
    if dt is None:
        return "         "
    if isinstance(dt, int):
        # Assume ms since epoch
        dt = datetime.utcfromtimestamp(dt / 1000)
    elif isinstance(dt, float):
        dt = datetime.utcfromtimestamp(dt / 1000)
    # Show as "MM-DD HH:MM:SS"
    return dt.strftime("%m-%d %H:%M:%S")

for symbol, info in side_by_side.items():
    print(f"{symbol}:")
    # Print table header
    print(f"{'STAGE':<20} {'FIRST TIME':<16} {'LAST TIME':<16} {'NULL COUNT':<10}")
    print("-" * 54)
    for stage, details in info.items():
        first_str = format_dt(details.get('first_dt'))
        last_str = format_dt(details.get('last_dt'))
        null_count = details.get('null_count', "")
        print(f"{stage.upper():<20} {first_str:<16} {last_str:<16} {null_count:<10}")
    print()


=== Side-by-side summary ===
BTC-USD-260626.OK:
STAGE                FIRST TIME       LAST TIME        NULL COUNT
------------------------------------------------------
TRIM_STRIP           09-01 00:00:00   09-30 23:59:59   0         
BIN_FF               09-01 00:00:09   09-30 23:59:59   0         
FINALIZE_BINNING     09-01 00:00:10   09-30 23:59:50   0         
SPREAD               09-01 00:00:10   09-30 23:59:50   0         
REL_SPREAD           09-01 00:00:10   09-30 23:59:50   0         
TENOR                09-01 00:00:10   09-30 23:59:50   0         
EARLY_ROLL           09-01 00:00:10   09-30 23:59:50   0         
LOG                  09-01 00:00:10   09-30 23:59:50   0         

BTC-USD-251031.OK:
STAGE                FIRST TIME       LAST TIME        NULL COUNT
------------------------------------------------------
TRIM_STRIP           09-01 00:00:00   09-30 23:59:59   0         
BIN_FF               09-01 00:00:09   09-30 23:59:59   0         
FINALIZE_BINNING     09-01 00